# ANNIE Co-primary μ⁻ / π⁺ Cherenkov Analysis
**Geant4 FTFP_BERT | Shuffled 30k sample**

Notebook structure:
1. **Imports** — all packages
2. **Load ROOT** — open file, read EventTree + helpers
3. **Definitions** — physics constants, helper functions, signal filter
4. **Computations** — derived columns, masks, StepTree accumulators
5. **Plots** — one cell per figure (labels easy to edit)

Export PNGs are included in each plot cell but **commented out** (`# fig.savefig(...)`).  
Un-comment a line to write a 600 dpi PNG next to the ROOT file.

---
## Cell 1 — Imports

In [ ]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from IPython.display import display, Markdown

plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Imports OK')

---
## Cell 2 — Load ROOT file

In [ ]:
# ── Path to ROOT file ─────────────────────────────────────────────────────────
SHUF_PATH = Path(r"D:\research\ANNIE\simualtion\my_geant4\current040126\annie_coprimary_30k_shuffled.root")

# PNG export directory — same folder as the ROOT file (un-comment savefig lines to use)
PNG_DIR = SHUF_PATH.parent

# ── Helper: pick tree with most entries ───────────────────────────────────────
def choose_best_tree(root_file, tree_name):
    matches = []
    for key in root_file.keys():
        if key.split(";")[0] == tree_name:
            obj = root_file[key]
            n = getattr(obj, "num_entries", -1)
            matches.append((key, n))
    if not matches:
        return None
    matches.sort(key=lambda t: t[1], reverse=True)
    return root_file[matches[0][0]]

# ── Quick file check ──────────────────────────────────────────────────────────
with uproot.open(SHUF_PATH) as f:
    print("Keys found:")
    for k in sorted(f.keys()):
        print(f"  {k}")

print("ROOT file OK")

---
## Cell 3 — Definitions (constants, helpers, Frank-Tamm, signal filter)

In [ ]:
# ── Detector geometry ─────────────────────────────────────────────────────────
R_CYL        = 1600.0          # mm, ANNIE inner radius
Z_MAX        = 2000.0          # mm, half-height
DET_EFF      = 0.025           # PMT geometric efficiency
C_LIGHT      = 299.792         # mm/ns
N_WATER      = 1.34
C_WATER      = C_LIGHT / N_WATER
MU_P_THRESHOLD = 0.1185        # GeV/c — muon MRD threshold
MAX_PHYSICAL_CM = 512.0        # ANNIE cylinder diagonal (cm)

# ── Particle masses (GeV/c²) ──────────────────────────────────────────────────
M_MU  = 0.1056583755
M_PI  = 0.13957039
ALPHA = 1 / 137.035999084

# ── Frank-Tamm wavelength grid (300–700 nm) ───────────────────────────────────
LAM_MIN_NM  = 300.0
LAM_MAX_NM  = 700.0
N_LAM_BINS  = 40
LAM_EDGES_NM = np.linspace(LAM_MIN_NM, LAM_MAX_NM, N_LAM_BINS + 1)
LAM_CENT_NM  = 0.5 * (LAM_EDGES_NM[:-1] + LAM_EDGES_NM[1:])
DLAM_CM      = np.diff(LAM_EDGES_NM) * 1e-7
LAM_CENT_CM  = LAM_CENT_NM * 1e-7
FT_LIMIT     = 387.0           # Frank-Tamm β→1 in water, ph/cm

# ── Colour palette ────────────────────────────────────────────────────────────
C_MU    = '#1f77b4'   # blue   — muon overall
C_PI    = '#d62728'   # red    — pion
C_MC    = '#aec7e8'   # lt blue — contained muon
C_MU2   = '#ff7f0e'   # orange — punch-through muon
C_DECAY = '#2ca02c'   # green  — decay
C_INEL  = '#9467bd'   # purple — inelastic
C_OTHER = '#8c564b'   # brown  — escapes/other

# ── Geometry helper ───────────────────────────────────────────────────────────
def tof_to_cylinder(x, y, z):
    """Shortest distance (mm) from point to ANNIE cylinder wall."""
    r        = np.sqrt(x**2 + y**2)
    d_barrel = np.clip(R_CYL - r, 0, None)
    d_endcap = np.clip(np.where(z >= 0, Z_MAX - z, Z_MAX + z), 0, None)
    return np.minimum(d_barrel, d_endcap)

# ── Relativistic kinematics ───────────────────────────────────────────────────
def beta_from_p(p, m):
    p = np.asarray(p, dtype=float)
    return p / np.sqrt(p**2 + m**2)

def gamma_from_p(p, m):
    p = np.asarray(p, dtype=float)
    return np.sqrt(1.0 + (p / m)**2)

def effective_threshold_momentum(m):
    n_eff   = np.average(n_water_dispersion(LAM_CENT_NM), weights=1./LAM_CENT_CM**2)
    beta_thr = 1.0 / n_eff
    gamma_thr = 1.0 / np.sqrt(1.0 - beta_thr**2)
    return m * beta_thr * gamma_thr, n_eff

# ── Water dispersion (Daimon & Masumura Sellmeier) ────────────────────────────
def n_water_dispersion(lam_nm):
    lam  = np.asarray(lam_nm, dtype=float) * 1e-3
    lam2 = lam**2
    B = np.array([0.75831, 0.08495, 0.00143])
    C = np.array([0.01007, 0.08997, 897.0  ])
    n2 = 1.0 + np.sum(B[:, None] * lam2[None, :]
                      / (lam2[None, :] - C[:, None]), axis=0)
    return np.sqrt(n2)

# ── Frank-Tamm instantaneous dN/dx (ph/cm) ───────────────────────────────────
def frank_tamm_per_cm_band(p, m):
    beta   = beta_from_p(p, m)
    beta2  = np.atleast_1d(beta)**2
    nlam   = n_water_dispersion(LAM_CENT_NM)
    headroom = np.clip(1.0 - 1.0 / (beta2[:, None] * nlam[None, :]**2), 0., None)
    d2N    = 2.0 * np.pi * ALPHA * headroom / (LAM_CENT_CM[None, :]**2)
    return np.sum(d2N * DLAM_CM[None, :], axis=1)

# ── Bethe-Bloch dE/dx in water (MeV/cm) ──────────────────────────────────────
def bethe_bloch_water_GeV(p_GeV, m_GeV):
    p_MeV  = np.asarray(p_GeV, dtype=float) * 1000.
    M_MeV  = m_GeV * 1000.
    E_MeV  = np.sqrt(p_MeV**2 + M_MeV**2)
    gamma  = E_MeV / M_MeV
    beta   = p_MeV / E_MeV
    bg     = beta * gamma
    I_water   = 79.7e-6
    Z_over_A  = 0.555
    me        = 0.511
    K         = 0.307075
    x         = np.log10(bg)
    C_s, x0, x1, a, k = -3.5017, 0.2400, 2.8004, 0.09116, 3.4773
    delta = np.where(x >= x1,
                2*np.log(10)*x + C_s,
                np.where(x >= x0,
                         2*np.log(10)*x + C_s + a*(x1 - x)**k,
                         0.0))
    Tmax  = (2*me*beta**2*gamma**2
             / (1 + 2*gamma*me/M_MeV + (me/M_MeV)**2))
    ln_t  = np.log(2*me*beta**2*gamma**2*Tmax / I_water**2)
    return K * Z_over_A / beta**2 * (0.5*ln_t - beta**2 - delta/2)

# ── Track-averaged Frank-Tamm (matches cher_total/track_len) ─────────────────
def frank_tamm_track_averaged(p0_array, m_GeV, n_steps=200):
    p0_array = np.asarray(p0_array, dtype=float)
    result   = np.empty_like(p0_array)
    n_eff    = np.average(n_water_dispersion(LAM_CENT_NM), weights=1./LAM_CENT_CM**2)
    beta_thr = 1.0 / n_eff
    p_thr    = m_GeV * beta_thr / np.sqrt(1.0 - beta_thr**2)
    for i, p0 in enumerate(p0_array):
        if p0 <= p_thr:
            result[i] = 0.0
            continue
        E0    = np.sqrt(p0**2 + m_GeV**2)
        E_min = max(np.sqrt(p_thr**2 + m_GeV**2) * 0.95, m_GeV * 1.001)
        E_grid = np.linspace(E0, E_min, n_steps)
        total_photons = 0.0
        total_length  = 0.0
        for j in range(len(E_grid) - 1):
            E_mid = 0.5 * (E_grid[j] + E_grid[j+1])
            dE    = abs(E_grid[j] - E_grid[j+1])
            p_mid = np.sqrt(max(E_mid**2 - m_GeV**2, 0.))
            if p_mid <= 0:
                continue
            dEdx_mev_cm = bethe_bloch_water_GeV(p_mid, m_GeV)
            if dEdx_mev_cm <= 0:
                continue
            dx_cm = (dE * 1000.) / dEdx_mev_cm
            dNdx  = frank_tamm_per_cm_band(np.array([p_mid]), m_GeV)[0]
            total_photons += dNdx * dx_cm
            total_length  += dx_cm
        result[i] = total_photons / total_length if total_length > 0 else 0.
    return result

# ── Binned quantiles ──────────────────────────────────────────────────────────
def binned_quantiles(x, y, bins, qs=(0.025, 0.16, 0.50, 0.84, 0.975), min_count=35):
    x = np.asarray(x); y = np.asarray(y)
    centers = 0.5 * (bins[:-1] + bins[1:])
    out     = {q: np.full(len(centers), np.nan) for q in qs}
    counts  = np.zeros(len(centers), dtype=int)
    idx     = np.digitize(x, bins) - 1
    for i in range(len(centers)):
        m = (idx == i) & np.isfinite(x) & np.isfinite(y)
        counts[i] = m.sum()
        if counts[i] >= min_count:
            for q in qs:
                out[q][i] = np.quantile(y[m], q)
    return centers, out, counts

# ── π⁺ fate simplifier ───────────────────────────────────────────────────────
def simplify_fate(p):
    if 'Decay'     in p: return 'Decay'
    if 'Inelastic' in p: return 'Inelastic'
    return 'Escapes'

# ── Signal filter constants ───────────────────────────────────────────────────
LATE_OUTLIER_EID = 23630      # single large-TOF outlier event

# ── EventTree columns needed ─────────────────────────────────────────────────
SIG_COLS = [
    'event_id',
    'mu_cher_total', 'mu_track_len', 'mu_time_stop', 'mu_contained',
    'pi_cher_total', 'pi_track_len', 'pi_time_end',  'pi_contained',
    'mu_p0', 'pi_p0', 'open_angle_deg', 'Enu',
    'mu_final_process', 'mu_KE0',
    'pi_final_process', 'pi_track_id',
    'n_secondary_pi',
    'smu_t_start', 'smu_KE0', 'smu_time_stop', 'smu_contained',
    'smu_cher_total',
    'michel_e_present', 'michel_e_time_start', 'michel_e_KE0',
    'michel_e_cher_total',
]

print('Definitions OK')

---
## Cell 4 — Computations (load data, derived columns, signal filter, StepTree prep)

In [ ]:
# ── 4a: Load EventTree ────────────────────────────────────────────────────────
with uproot.open(SHUF_PATH) as f:
    et_key  = sorted([k for k in f.keys() if 'EventTree' in k])[-1]
    ev_all  = f[et_key].arrays(SIG_COLS, library='pd')

print(f"Loaded {len(ev_all):,} events from {et_key}")

# ── 4b: Type fixes ────────────────────────────────────────────────────────────
ev_all['mu_contained']    = ev_all['mu_contained'].astype(bool)
ev_all['pi_contained']    = ev_all['pi_contained'].astype(bool)
ev_all['michel_e_present']= ev_all['michel_e_present'].astype(bool)
ev_all['smu_contained']   = ev_all['smu_contained'].astype(bool)

for col in ['mu_final_process', 'pi_final_process']:
    ev_all[col] = ev_all[col].astype(str).str.strip().str.rstrip('\x00')

# ── 4c: Unit fix — track_len stored in mm, convert to cm ─────────────────────
if ev_all['mu_track_len'].max() > MAX_PHYSICAL_CM * 2:
    ev_all['mu_track_len'] = ev_all['mu_track_len'] / 10.0
    ev_all['pi_track_len'] = ev_all['pi_track_len'] / 10.0
    print("  track_len branches were in mm — converted to cm")

# ── 4d: Derived columns (all events) ─────────────────────────────────────────
ev_all['fate']         = ev_all['pi_final_process'].apply(simplify_fate)
ev_all['mu_cher_per_cm'] = np.where(ev_all['mu_track_len'] > 0,
                                    ev_all['mu_cher_total'] / ev_all['mu_track_len'], np.nan)
ev_all['pi_cher_per_cm'] = np.where(ev_all['pi_track_len'] > 0,
                                    ev_all['pi_cher_total'] / ev_all['pi_track_len'], np.nan)

# ── 4e: Containment masks ─────────────────────────────────────────────────────
mu_cont   = ev_all[ev_all['mu_contained']]
mu_uncont = ev_all[~ev_all['mu_contained']]

# ── 4f: Signal filter (MRD punch-through, above threshold) ───────────────────
cut_threshold = ev_all['mu_p0'] > MU_P_THRESHOLD
cut_punchthru = ~ev_all['mu_contained']
cut_signal    = cut_threshold & cut_punchthru

ev_sig = ev_all[cut_signal].copy().set_index('event_id')
ev_sig['ratio']           = ev_sig['pi_cher_total']  / ev_sig['mu_cher_total']
ev_sig['mu_cher_per_cm']  = ev_sig['mu_cher_total']  / ev_sig['mu_track_len']
ev_sig['pi_cher_per_cm']  = ev_sig['pi_cher_total']  / ev_sig['pi_track_len']
ev_sig['dt_pi_after_mu']  = ev_sig['pi_time_end']    - ev_sig['mu_time_stop']
ev_sig['mu_plus_lifetime']= np.where(
    ev_sig['smu_t_start'] > 0,
    ev_sig['smu_time_stop'] - ev_sig['smu_t_start'],
    np.nan
)

# ── 4g: True / fake Michel classification ────────────────────────────────────
ev_sig['true_michel'] = (
    ev_sig['michel_e_present'] &
    (ev_sig['fate'] == 'Decay') &
    (ev_sig['smu_t_start'] > 0) &
    (ev_sig['n_secondary_pi'] == 0)
)
ev_sig['fake_michel'] = (
    ev_sig['michel_e_present'] &
    (ev_sig['n_secondary_pi'] > 0)
)

# ── 4h: Convenience sets / lookups used by StepTree cells ────────────────────
sig_ids      = set(np.int32(list(ev_sig.index)))
decay_ids    = set(np.int32(
    ev_sig[ev_sig['true_michel'] | (ev_sig['fate']=='Decay')].index.tolist()))
pi_tid_lookup = ev_sig['pi_track_id'].to_dict()

# muon exit time for shoulder plot
ev_sig['mu_water_exit_ns'] = ev_sig['mu_time_stop']
mu_exit_lookup = ev_sig[['mu_water_exit_ns']]

# outlier-removed copy
ev_sig_no23630 = ev_sig[ev_sig.index != LATE_OUTLIER_EID].copy()

# ── Summary ───────────────────────────────────────────────────────────────────
n_total     = len(ev_all)
n_subthresh = (~cut_threshold).sum()
n_contained = cut_threshold.sum() - cut_signal.sum()
n_signal    = len(ev_sig)

print('='*60)
print(' SIGNAL FILTER SUMMARY')
print('='*60)
print(f"  Total events           : {n_total:,}")
print(f"  Sub-threshold muon     : {n_subthresh:,} ({n_subthresh/n_total*100:.1f}%)")
print(f"  Contained muon (stops) : {n_contained:,} ({n_contained/n_total*100:.1f}%)")
print(f"  Signal events          : {n_signal:,} ({n_signal/n_total*100:.1f}%)")
print()
print(f"  μ⁻ containment fraction  : {ev_all['mu_contained'].mean()*100:.1f}%")
print(f"  π⁺ containment fraction  : {ev_all['pi_contained'].mean()*100:.1f}%")
print(f"  True Michel events (sig) : {ev_sig['true_michel'].sum():,}")
print(f"  Outlier event {LATE_OUTLIER_EID} excluded in ev_sig_no23630")

---
## Plot 1 — Prompt light yield + timing (2×2 panel)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Prompt μ⁻ and π⁺ Light Yield + Timing | Shuffled 30k sample',
             fontsize=13, fontweight='bold')

# ── Panel A: Total Cherenkov yield ────────────────────────────────────────────
ax = axes[0, 0]
bins_y = np.linspace(0, ev_all['mu_cher_total'].quantile(0.995), 80)
ax.hist(mu_cont['mu_cher_total'],   bins=bins_y, color=C_MC,  alpha=0.75,
        label=f'μ⁻ contained (n={len(mu_cont):,})')
ax.hist(mu_uncont['mu_cher_total'], bins=bins_y, color=C_MU2, alpha=0.65,
        label=f'μ⁻ punch-through (n={len(mu_uncont):,})')
ax.axvline(ev_all['mu_cher_total'].mean(), color=C_MU, lw=1.8, ls='--',
           label=f'μ⁻ mean = {ev_all["mu_cher_total"].mean():,.0f}')
ax.set_xlabel('Total Cherenkov photons — μ⁻')
ax.set_ylabel('Events')
ax.set_title('A  μ⁻ prompt Cherenkov yield\n(contained vs punch-through)')
ax.legend(fontsize=9)
ax2 = ax.twinx()
bins_pi = np.linspace(0, ev_all['pi_cher_total'].quantile(0.995), 80)
ax2.hist(ev_all['pi_cher_total'], bins=bins_pi, color=C_PI, alpha=0.35,
         label=f'π⁺ (n={len(ev_all):,})')
ax2.axvline(ev_all['pi_cher_total'].mean(), color=C_PI, lw=1.8, ls='--',
            label=f'π⁺ mean = {ev_all["pi_cher_total"].mean():,.0f}')
ax2.set_ylabel('Events — π⁺ (right axis)', color=C_PI)
ax2.tick_params(axis='y', labelcolor=C_PI)
ax2.legend(fontsize=9, loc='upper right')

# ── Panel B: Time to stop / exit ──────────────────────────────────────────────
ax = axes[0, 1]
t_max  = max(ev_all['mu_time_stop'].quantile(0.995),
             ev_all['pi_time_end'].quantile(0.995))
bins_t = np.linspace(0, t_max, 80)
ax.hist(mu_cont['mu_time_stop'],   bins=bins_t, color=C_MC,  alpha=0.75, label='μ⁻ contained')
ax.hist(mu_uncont['mu_time_stop'], bins=bins_t, color=C_MU2, alpha=0.65, label='μ⁻ punch-through')
ax.hist(ev_all['pi_time_end'],     bins=bins_t, color=C_PI,  alpha=0.45,
        label=f'π⁺  mean={ev_all["pi_time_end"].mean():.2f} ns')
ax.axvline(ev_all['mu_time_stop'].mean(), color=C_MU, lw=1.8, ls='--',
           label=f'μ⁻ mean = {ev_all["mu_time_stop"].mean():.2f} ns')
ax.set_xlabel('Time to stop / exit cylinder (ns)')
ax.set_ylabel('Events')
ax.set_title('B  Time inside cylinder\nμ⁻ stop / exit time vs π⁺ end time')
ax.legend(fontsize=9)

# ── Panel C: Track length ─────────────────────────────────────────────────────
ax = axes[1, 0]
len_max = max(ev_all['mu_track_len'].quantile(0.995),
              ev_all['pi_track_len'].quantile(0.995))
bins_l  = np.linspace(0, len_max, 80)
ax.hist(mu_cont['mu_track_len'],   bins=bins_l, color=C_MC,  alpha=0.75,
        label=f'μ⁻ contained  mean={mu_cont["mu_track_len"].mean():.1f} cm')
ax.hist(mu_uncont['mu_track_len'], bins=bins_l, color=C_MU2, alpha=0.65,
        label=f'μ⁻ punch-through  mean={mu_uncont["mu_track_len"].mean():.1f} cm')
ax.hist(ev_all['pi_track_len'],    bins=bins_l, color=C_PI,  alpha=0.45,
        label=f'π⁺  mean={ev_all["pi_track_len"].mean():.1f} cm')
ax.set_xlabel('Track length inside ANNIE (cm)')
ax.set_ylabel('Events')
ax.set_title('C  Track length inside cylinder')
ax.legend(fontsize=9)

# ── Panel D: dN_Cherenkov/dx vs Frank-Tamm limit ──────────────────────────────
ax = axes[1, 1]
valid_mu = ev_all['mu_cher_per_cm'].dropna()
valid_pi = ev_all['pi_cher_per_cm'].dropna()
dndx_max = max(valid_mu.quantile(0.995), valid_pi.quantile(0.995), FT_LIMIT)
bins_d   = np.linspace(0, dndx_max, 80)
ax.hist(valid_mu, bins=bins_d, color=C_MU, alpha=0.55,
        label=f'μ⁻  mean={valid_mu.mean():.1f} ph/cm')
ax.hist(valid_pi, bins=bins_d, color=C_PI, alpha=0.55,
        label=f'π⁺  mean={valid_pi.mean():.1f} ph/cm')
ax.axvline(FT_LIMIT, color='black', lw=2, ls=':',
           label=f'Frank-Tamm β→1 limit = {FT_LIMIT:.0f} ph/cm')
ax.set_xlabel('Cherenkov photons per cm of track')
ax.set_ylabel('Events')
ax.set_title('D  dN_Cherenkov/dx (β-averaged over full track)\nvs Frank-Tamm β→1 limit')
ax.legend(fontsize=9)

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot1_prompt_yield_timing.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 2 — π⁺/μ⁻ Cherenkov ratio vs opening angle (1×3 panel)

In [ ]:
# Re-load just the columns needed
angle_cols = ['open_angle_deg', 'Enu', 'mu_p0', 'pi_p0',
              'mu_cher_total', 'pi_cher_total', 'mu_contained', 'pi_contained']
with uproot.open(SHUF_PATH) as f:
    et_key = sorted([k for k in f.keys() if 'EventTree' in k])[-1]
    ev2    = f[et_key].arrays(angle_cols, library='pd')
ev2['mu_contained'] = ev2['mu_contained'].astype(bool)
ev2['pi_contained'] = ev2['pi_contained'].astype(bool)
ev2['ratio']        = np.where(ev2['mu_cher_total'] > 0,
                               ev2['pi_cher_total'] / ev2['mu_cher_total'], np.nan)

angle_bins  = np.arange(0, 181, 10)
angle_cents = (angle_bins[:-1] + angle_bins[1:]) / 2
ev2['angle_bin'] = pd.cut(ev2['open_angle_deg'], bins=angle_bins, labels=angle_cents)
bin_stats = (ev2.groupby('angle_bin', observed=True)['ratio']
             .agg(mean='mean', median='median', std='std', count='count')
             .reset_index())
bin_stats['se'] = bin_stats['std'] / np.sqrt(bin_stats['count'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('π⁺/μ⁻ Cherenkov Ratio vs Opening Angle | Shuffled 30k sample',
             fontsize=13, fontweight='bold')

# ── Panel A: Opening angle distribution ──────────────────────────────────────
ax = axes[0]
ax.hist(ev2['open_angle_deg'], bins=60, color='#7f7f7f', alpha=0.75)
ax.axvline(ev2['open_angle_deg'].mean(),   color='black', lw=1.8, ls='--',
           label=f'mean = {ev2["open_angle_deg"].mean():.1f}°')
ax.axvline(ev2['open_angle_deg'].median(), color='black', lw=1.8, ls=':',
           label=f'median = {ev2["open_angle_deg"].median():.1f}°')
ax.set_xlabel('μ⁻ / π⁺ opening angle (degrees)')
ax.set_ylabel('Events')
ax.set_title('A  Opening angle distribution')
ax.legend(fontsize=9)

# ── Panel B: Mean ratio per angle bin ────────────────────────────────────────
ax = axes[1]
x  = bin_stats['angle_bin'].astype(float)
ax.errorbar(x, bin_stats['mean'], yerr=bin_stats['se'],
            fmt='o-', color=C_PI, lw=1.8, ms=5, capsize=3, label='mean π⁺/μ⁻ ± SE')
ax.fill_between(x, bin_stats['mean'] - bin_stats['std'],
                   bin_stats['mean'] + bin_stats['std'],
                alpha=0.15, color=C_PI, label='±1σ band')
ax.axhline(ev2['ratio'].mean(), color='black', lw=1.5, ls='--',
           label=f'overall mean = {ev2["ratio"].mean():.3f}')
ax.set_xlabel('Opening angle (degrees)')
ax.set_ylabel('π⁺/μ⁻ Cherenkov ratio')
ax.set_title('B  Ratio vs opening angle\n(does separation improve at large angles?)')
ax.legend(fontsize=9)

# ── Panel C: 2D hexbin ────────────────────────────────────────────────────────
ax = axes[2]
valid = ev2.dropna(subset=['ratio', 'open_angle_deg'])
hb = ax.hexbin(valid['open_angle_deg'], valid['ratio'],
               gridsize=40, cmap='viridis', mincnt=1,
               extent=[0, 180, 0, valid['ratio'].quantile(0.995)])
plt.colorbar(hb, ax=ax, label='Events per bin')
ax.axhline(ev2['ratio'].mean(), color='white', lw=1.5, ls='--',
           label=f'mean ratio = {ev2["ratio"].mean():.3f}')
ax.set_xlabel('Opening angle (degrees)')
ax.set_ylabel('π⁺/μ⁻ Cherenkov ratio')
ax.set_title('C  2D density: ratio vs angle')
ax.legend(fontsize=9, labelcolor='white')

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot2_ratio_vs_angle.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 3 — π⁺ kinematics, fate, and ratio vs opening angle (3×3, signal sample)

In [ ]:
with uproot.open(SHUF_PATH) as f:
    et_key   = sorted([k for k in f.keys() if 'EventTree' in k])[-1]
    extra_cols = ['pi_final_process', 'mu_p0', 'pi_p0',
                  'mu_cher_total', 'pi_cher_total', 'open_angle_deg',
                  'mu_contained', 'mu_KE0']
    ev_extra = f[et_key].arrays(extra_cols, library='pd')
ev_extra['mu_contained']    = ev_extra['mu_contained'].astype(bool)
ev_extra['pi_final_process']= (ev_extra['pi_final_process']
                               .astype(str).str.strip().str.rstrip('\x00'))
# Apply same signal filter
sig_mask = (ev_extra['mu_p0'] > MU_P_THRESHOLD) & (~ev_extra['mu_contained'])
ev_extra = ev_extra[sig_mask].copy()
ev_extra['fate']  = ev_extra['pi_final_process'].apply(simplify_fate)
ev_extra['ratio'] = ev_extra['pi_cher_total'] / ev_extra['mu_cher_total']

fate_colors  = {'Decay': C_DECAY, 'Inelastic': C_INEL, 'Escapes': C_OTHER}
angle_bins   = np.arange(0, 181, 10)
angle_cents  = (angle_bins[:-1] + angle_bins[1:]) / 2

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle('π⁺ Kinematics, Fate, and Ratio vs Opening Angle | MRD Signal Sample',
             fontsize=13, fontweight='bold')

# ── Row 0: momenta ────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.hist(ev_extra['pi_p0'] * 1000, bins=60, color=C_PI, alpha=0.75)
ax.axvline(156.5, color='black', lw=1.8, ls='--', label='π⁺ threshold 156.5 MeV/c')
ax.axvline(ev_extra['pi_p0'].mean() * 1000, color=C_PI, lw=1.8, ls=':',
           label=f'mean = {ev_extra["pi_p0"].mean()*1000:.1f} MeV/c')
ax.set_xlabel('π⁺ initial momentum (MeV/c)'); ax.set_ylabel('Events')
ax.set_title('A  π⁺ momentum distribution (signal sample)'); ax.legend(fontsize=9)

ax = axes[0, 1]
hb = ax.hexbin(ev_extra['open_angle_deg'], ev_extra['pi_p0']*1000,
               gridsize=40, cmap='viridis', mincnt=1)
plt.colorbar(hb, ax=ax, label='Events per bin')
ax.axhline(156.5, color='white', lw=1.8, ls='--', label='π⁺ threshold')
pi_p0_bins = (ev_extra.groupby(
    pd.cut(ev_extra['open_angle_deg'], bins=angle_bins, labels=angle_cents), observed=True
)['pi_p0'].median() * 1000)
ax.plot(pi_p0_bins.index.astype(float), pi_p0_bins.values,
        'o-', color='white', lw=1.8, ms=4, label='median π⁺ p₀')
ax.set_xlabel('Opening angle (degrees)'); ax.set_ylabel('π⁺ initial momentum (MeV/c)')
ax.set_title('B  π⁺ momentum vs opening angle'); ax.legend(fontsize=9)

ax = axes[0, 2]
hb = ax.hexbin(ev_extra['open_angle_deg'], ev_extra['mu_p0']*1000,
               gridsize=40, cmap='plasma', mincnt=1)
plt.colorbar(hb, ax=ax, label='Events per bin')
mu_p0_bins = (ev_extra.groupby(
    pd.cut(ev_extra['open_angle_deg'], bins=angle_bins, labels=angle_cents), observed=True
)['mu_p0'].median() * 1000)
ax.plot(mu_p0_bins.index.astype(float), mu_p0_bins.values,
        'o-', color='white', lw=1.8, ms=4, label='median μ⁻ p₀')
ax.set_xlabel('Opening angle (degrees)'); ax.set_ylabel('μ⁻ initial momentum (MeV/c)')
ax.set_title('C  μ⁻ momentum vs opening angle'); ax.legend(fontsize=9)

# ── Row 1: fate ───────────────────────────────────────────────────────────────
ax = axes[1, 0]
for fate, grp in ev_extra.groupby('fate'):
    ax.hist(grp['open_angle_deg'], bins=angle_bins, alpha=0.6,
            color=fate_colors.get(fate, 'grey'), label=f'{fate} (n={len(grp):,})')
ax.set_xlabel('Opening angle (degrees)'); ax.set_ylabel('Events')
ax.set_title('D  π⁺ fate vs opening angle (counts)'); ax.legend(fontsize=9)

ax = axes[1, 1]
decay_frac, inel_frac, other_frac = [], [], []
for lo, hi in zip(angle_bins[:-1], angle_bins[1:]):
    sl = ev_extra[(ev_extra['open_angle_deg'] >= lo) & (ev_extra['open_angle_deg'] < hi)]
    n  = len(sl)
    if n > 0:
        decay_frac.append((sl['fate']=='Decay').sum()     / n)
        inel_frac.append( (sl['fate']=='Inelastic').sum() / n)
        other_frac.append((sl['fate']=='Escapes').sum()   / n)
    else:
        decay_frac.append(0); inel_frac.append(0); other_frac.append(0)
ax.bar(angle_cents, decay_frac, width=9, color=C_DECAY, alpha=0.8, label='Decay')
ax.bar(angle_cents, inel_frac,  width=9, color=C_INEL,  alpha=0.8, bottom=decay_frac, label='Inelastic')
ax.bar(angle_cents, other_frac, width=9, color=C_OTHER, alpha=0.8,
       bottom=np.array(decay_frac)+np.array(inel_frac), label='Escapes')
ax.set_xlabel('Opening angle (degrees)'); ax.set_ylabel('Fraction')
ax.set_title('E  π⁺ fate fraction vs opening angle'); ax.legend(fontsize=9)

ax = axes[1, 2]
for fate, grp in ev_extra.groupby('fate'):
    ratio_bins = grp.groupby(
        pd.cut(grp['open_angle_deg'], bins=angle_bins, labels=angle_cents), observed=True
    )['ratio'].median()
    ax.plot(ratio_bins.index.astype(float), ratio_bins.values,
            'o-', color=fate_colors.get(fate, 'grey'), lw=1.8, ms=4, label=fate)
ax.set_xlabel('Opening angle (degrees)'); ax.set_ylabel('Median π⁺/μ⁻ ratio')
ax.set_title('F  Median ratio vs angle by π⁺ fate'); ax.legend(fontsize=9)

# ── Row 2: ratio vs μ⁻ momentum ───────────────────────────────────────────────
ax = axes[2, 0]
mu_p_bins_edges = np.percentile(ev_extra['mu_p0']*1000, np.linspace(0,100,11))
ev_extra['mu_p_bin'] = pd.cut(ev_extra['mu_p0']*1000, bins=mu_p_bins_edges)
ratio_vs_mup = (ev_extra.groupby('mu_p_bin', observed=True)['ratio']
                .agg(['median','mean','std']).reset_index())
cents = [(iv.left+iv.right)/2 for iv in ratio_vs_mup['mu_p_bin']]
ax.plot(cents, ratio_vs_mup['median'], 'o-', color=C_MU,  lw=1.8, ms=5, label='median ratio')
ax.plot(cents, ratio_vs_mup['mean'],   's--', color=C_MU2, lw=1.5, ms=4, label='mean ratio')
ax.set_xlabel('μ⁻ initial momentum (MeV/c)'); ax.set_ylabel('π⁺/μ⁻ Cherenkov ratio')
ax.set_title('G  Ratio vs μ⁻ momentum'); ax.legend(fontsize=9)

ax = axes[2, 1]
hb = ax.hexbin(ev_extra['mu_p0']*1000,
               ev_extra['ratio'].clip(upper=ev_extra['ratio'].quantile(0.98)),
               gridsize=40, cmap='viridis', mincnt=1)
plt.colorbar(hb, ax=ax, label='Events per bin')
ax.set_xlabel('μ⁻ initial momentum (MeV/c)'); ax.set_ylabel('π⁺/μ⁻ Cherenkov ratio')
ax.set_title('H  2D density: ratio vs μ⁻ momentum')

ax = axes[2, 2]
ax.hist2d(ev_extra['open_angle_deg'], ev_extra['pi_p0']*1000,
          bins=[angle_bins, 60], cmap='viridis')
ax.set_xlabel('Opening angle (degrees)'); ax.set_ylabel('π⁺ initial momentum (MeV/c)')
ax.set_title('I  2D: opening angle vs π⁺ momentum')

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot3_kinematics_fate_angle.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 4 — μ⁻ / π⁺ momentum, β, γ comparison (1×4 panel)

In [ ]:
BETA_THR  = 1.0 / N_WATER
GAMMA_THR = 1.0 / np.sqrt(1.0 - BETA_THR**2)
PTHR_MU   = M_MU * BETA_THR * GAMMA_THR
PTHR_PI   = M_PI * BETA_THR * GAMMA_THR

ev_sig['mu_beta']   = beta_from_p(ev_sig['mu_p0'], M_MU)
ev_sig['pi_beta']   = beta_from_p(ev_sig['pi_p0'], M_PI)
ev_sig['mu_gamma']  = gamma_from_p(ev_sig['mu_p0'], M_MU)
ev_sig['pi_gamma']  = gamma_from_p(ev_sig['pi_p0'], M_PI)

def cherenkov_proxy(beta, n=N_WATER):
    val = 1.0 - 1.0 / (beta**2 * n**2)
    return np.where(beta * n > 1, val, 0.0)

ev_sig['mu_cher_proxy'] = cherenkov_proxy(ev_sig['mu_beta'])
ev_sig['pi_cher_proxy'] = cherenkov_proxy(ev_sig['pi_beta'])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Primary μ⁻ vs π⁺ — momentum / β / γ / Cherenkov headroom | Signal sample',
             fontsize=13, fontweight='bold')

# Panel A: momentum
ax = axes[0]
ax.hist(ev_sig['mu_p0']*1000, bins=60, color=C_MU, alpha=0.6,
        label=f'μ⁻  mean={ev_sig["mu_p0"].mean()*1000:.1f} MeV/c')
ax.hist(ev_sig['pi_p0']*1000, bins=60, color=C_PI, alpha=0.6,
        label=f'π⁺  mean={ev_sig["pi_p0"].mean()*1000:.1f} MeV/c')
ax.axvline(PTHR_MU*1000, color=C_MU, lw=1.5, ls=':', label=f'μ⁻ thr={PTHR_MU*1000:.0f} MeV/c')
ax.axvline(PTHR_PI*1000, color=C_PI, lw=1.5, ls=':', label=f'π⁺ thr={PTHR_PI*1000:.0f} MeV/c')
ax.set_xlabel('Initial momentum (MeV/c)'); ax.set_ylabel('Events')
ax.set_title('A  Momentum distributions'); ax.legend(fontsize=9)

# Panel B: beta
ax = axes[1]
ax.hist(ev_sig['mu_beta'], bins=60, color=C_MU, alpha=0.6, label='μ⁻')
ax.hist(ev_sig['pi_beta'], bins=60, color=C_PI, alpha=0.6, label='π⁺')
ax.axvline(BETA_THR, color='black', lw=1.8, ls='--', label=f'β_thr = 1/n = {BETA_THR:.4f}')
ax.set_xlabel('β = v/c'); ax.set_ylabel('Events')
ax.set_title('B  β distributions'); ax.legend(fontsize=9)

# Panel C: gamma
ax = axes[2]
ax.hist(ev_sig['mu_gamma'], bins=60, color=C_MU, alpha=0.6, label='μ⁻')
ax.hist(ev_sig['pi_gamma'], bins=60, color=C_PI, alpha=0.6, label='π⁺')
ax.set_xlabel('Lorentz γ'); ax.set_ylabel('Events')
ax.set_title('C  γ distributions'); ax.legend(fontsize=9)

# Panel D: Cherenkov headroom 1 - 1/(β²n²)
ax = axes[3]
ax.hist(ev_sig['mu_cher_proxy'], bins=60, color=C_MU, alpha=0.6,
        label=f'μ⁻  mean={ev_sig["mu_cher_proxy"].mean():.3f}')
ax.hist(ev_sig['pi_cher_proxy'], bins=60, color=C_PI, alpha=0.6,
        label=f'π⁺  mean={ev_sig["pi_cher_proxy"].mean():.3f}')
ax.set_xlabel('1 − 1/(β²n²)  [Cherenkov headroom]'); ax.set_ylabel('Events')
ax.set_title('D  Cherenkov headroom above threshold'); ax.legend(fontsize=9)

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot4_beta_gamma_headroom.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 5 — Analytic Cherenkov model validation (Frank-Tamm vs Geant4, 2×2)

In [ ]:
# Load needed columns from EventTree
ft_cols = ['event_id','mu_p0','pi_p0',
           'mu_cher_total','pi_cher_total',
           'mu_track_len','pi_track_len',
           'mu_final_process','pi_final_process']
with uproot.open(SHUF_PATH) as f:
    tree = choose_best_tree(f, 'EventTree')
    ev_ft = tree.arrays([c for c in ft_cols if c in tree.keys()], library='pd')

for c in ['mu_p0','pi_p0','mu_cher_total','pi_cher_total','mu_track_len','pi_track_len']:
    ev_ft[c] = pd.to_numeric(ev_ft[c], errors='coerce')
for c in ['mu_final_process','pi_final_process']:
    ev_ft[c] = ev_ft[c].astype(str).str.strip().str.rstrip('\x00')
if ev_ft['mu_track_len'].max() > 600:
    ev_ft['mu_track_len'] = ev_ft['mu_track_len'] / 10.0
    ev_ft['pi_track_len'] = ev_ft['pi_track_len'] / 10.0
ev_ft['mu_cher_per_cm'] = np.where(ev_ft['mu_track_len']>0, ev_ft['mu_cher_total']/ev_ft['mu_track_len'], np.nan)
ev_ft['pi_cher_per_cm'] = np.where(ev_ft['pi_track_len']>0, ev_ft['pi_cher_total']/ev_ft['pi_track_len'], np.nan)

mu_mask      = np.isfinite(ev_ft['mu_p0']) & np.isfinite(ev_ft['mu_cher_per_cm'])
pi_mask      = np.isfinite(ev_ft['pi_p0']) & np.isfinite(ev_ft['pi_cher_per_cm'])
pi_decay_mask = pi_mask & (ev_ft['pi_final_process'] == 'Decay')
pi_inel_mask  = pi_mask & ev_ft['pi_final_process'].str.contains('Inelastic', na=False)

pmax    = max(ev_ft['mu_p0'].quantile(0.995), ev_ft['pi_p0'].quantile(0.995))
p_grid  = np.linspace(0.001, pmax * 1.05, 300)
print('Computing track-averaged Frank-Tamm curves (~10 s)...')
mu_ft   = frank_tamm_track_averaged(p_grid, M_MU)
pi_ft   = frank_tamm_track_averaged(p_grid, M_PI)
PTHR_MU_EFF, _ = effective_threshold_momentum(M_MU)
PTHR_PI_EFF, _ = effective_threshold_momentum(M_PI)

p_bins  = np.linspace(0., pmax*1.05, 40)
mu_pc, mu_q, _ = binned_quantiles(ev_ft.loc[mu_mask,'mu_p0'], ev_ft.loc[mu_mask,'mu_cher_per_cm'], p_bins)
pi_pc, pi_q, _ = binned_quantiles(ev_ft.loc[pi_mask,'pi_p0'], ev_ft.loc[pi_mask,'pi_cher_per_cm'], p_bins)
pi_d_pc, pi_d_q, _ = binned_quantiles(ev_ft.loc[pi_decay_mask,'pi_p0'], ev_ft.loc[pi_decay_mask,'pi_cher_per_cm'], p_bins)
pi_i_pc, pi_i_q, _ = binned_quantiles(ev_ft.loc[pi_inel_mask,'pi_p0'],  ev_ft.loc[pi_inel_mask,'pi_cher_per_cm'],  p_bins)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analytic Frank-Tamm vs Geant4 dN_Cher/dx | Full 30k sample',
             fontsize=13, fontweight='bold')

def plot_panel(ax, pc, q, ft_p, ft_y, label, color, title):
    ax.fill_between(pc, q[0.025], q[0.975], alpha=0.15, color=color)
    ax.fill_between(pc, q[0.16],  q[0.84],  alpha=0.25, color=color)
    ax.plot(pc, q[0.50], 'o-', color=color, lw=1.8, ms=4, label=f'{label} median (Geant4)')
    ax.plot(ft_p, ft_y, '--', color='black', lw=2, label='Frank-Tamm (track-avg)')
    ax.set_xlabel('Initial momentum (GeV/c)'); ax.set_ylabel('dN/dx (ph/cm)')
    ax.set_title(title); ax.legend(fontsize=9)

plot_panel(axes[0,0], mu_pc, mu_q, p_grid, mu_ft, 'μ⁻', C_MU, 'A  μ⁻ dN/dx vs Frank-Tamm')
plot_panel(axes[0,1], pi_pc, pi_q, p_grid, pi_ft, 'π⁺ all', C_PI, 'B  π⁺ dN/dx vs Frank-Tamm')
plot_panel(axes[1,0], pi_d_pc, pi_d_q, p_grid, pi_ft, 'π⁺ decay', C_DECAY,
           'C  π⁺ decay only — matches Frank-Tamm')
plot_panel(axes[1,1], pi_i_pc, pi_i_q, p_grid, pi_ft, 'π⁺ inelastic', C_INEL,
           'D  π⁺ inelastic — truncated at high β → inflated dN/dx')

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot5_frank_tamm_validation.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 6 — Cherenkov cone angle vs β (Frank-Tamm, analytic)

In [ ]:
beta_grid = np.linspace(BETA_THR + 1e-5, 0.9999, 400)
n_eff     = np.average(n_water_dispersion(LAM_CENT_NM), weights=1./LAM_CENT_CM**2)
cos_theta = 1.0 / (beta_grid * n_eff)
cos_theta = np.clip(cos_theta, -1, 1)
theta_deg = np.degrees(np.arccos(cos_theta))

mu_beta_sig = beta_from_p(ev_sig['mu_p0'].values, M_MU)
pi_beta_sig = beta_from_p(ev_sig['pi_p0'].values, M_PI)

def cone_angle(beta):
    cos_t = np.clip(1.0/(np.asarray(beta)*n_eff), -1, 1)
    return np.degrees(np.arccos(cos_t))

mu_cone = cone_angle(np.clip(mu_beta_sig, BETA_THR+1e-8, 0.9999))
pi_cone = cone_angle(np.clip(pi_beta_sig, BETA_THR+1e-8, 0.9999))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Cherenkov Cone Angle | ANNIE water  n = 1.34',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(beta_grid, theta_deg, color='steelblue', lw=2, label='θ_C(β) analytic')
ax.axvline(BETA_THR, color='black', ls=':', lw=1.5, label=f'β_thr = {BETA_THR:.4f}')
ax.set_xlabel('β = v/c'); ax.set_ylabel('Cherenkov cone half-angle (°)')
ax.set_title('A  θ_C vs β (analytic)')
ax.set_xlim(0.74, 1.0); ax.legend(fontsize=9)

ax = axes[1]
ax.hist(mu_cone, bins=60, color=C_MU, alpha=0.7,
        label=f'μ⁻  mean={np.nanmean(mu_cone):.1f}°')
ax.hist(pi_cone, bins=60, color=C_PI, alpha=0.7,
        label=f'π⁺  mean={np.nanmean(pi_cone):.1f}°')
ax.set_xlabel('Cherenkov cone half-angle (°)'); ax.set_ylabel('Events')
ax.set_title('B  Cone angle distributions (signal sample)'); ax.legend(fontsize=9)

ax = axes[2]
ax.scatter(mu_beta_sig, mu_cone, s=3, alpha=0.15, color=C_MU, label='μ⁻')
ax.scatter(pi_beta_sig, pi_cone, s=3, alpha=0.15, color=C_PI, label='π⁺')
ax.plot(beta_grid, theta_deg, color='black', lw=1.5, ls='--', label='analytic')
ax.set_xlabel('β'); ax.set_ylabel('θ_C (°)')
ax.set_title('C  Cone angle vs β (scatter + theory)'); ax.legend(fontsize=9, markerscale=4)

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot6_cherenkov_cone_angle.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 7 — μ⁻ hump: stopping vs punch-through (1×3)

In [ ]:
with uproot.open(SHUF_PATH) as f:
    et_key  = sorted([k for k in f.keys() if 'EventTree' in k])[-1]
    ev_hump = f[et_key].arrays(
        ['event_id','mu_p0','mu_cher_total','mu_track_len','mu_contained'], library='pd')
ev_hump['mu_contained'] = ev_hump['mu_contained'].astype(bool)
for c in ['mu_p0','mu_cher_total','mu_track_len']:
    ev_hump[c] = pd.to_numeric(ev_hump[c], errors='coerce')
if ev_hump['mu_track_len'].max() > 600:
    ev_hump['mu_track_len'] = ev_hump['mu_track_len'] / 10.
ev_hump['mu_cher_per_cm'] = np.where(ev_hump['mu_track_len']>0,
                                     ev_hump['mu_cher_total']/ev_hump['mu_track_len'], np.nan)
stop_mask  = np.isfinite(ev_hump['mu_p0']) & np.isfinite(ev_hump['mu_cher_per_cm']) & ev_hump['mu_contained']
punch_mask = np.isfinite(ev_hump['mu_p0']) & np.isfinite(ev_hump['mu_cher_per_cm']) & ~ev_hump['mu_contained']

pmax_mu   = ev_hump['mu_p0'].quantile(0.995)
p_grid_mu = np.linspace(0.001, pmax_mu*1.05, 300)
print('Computing Frank-Tamm for μ⁻ hump plot (~5 s)...')
mu_ft_h   = frank_tamm_track_averaged(p_grid_mu, M_MU)
PTHR_MU_EFF, _ = effective_threshold_momentum(M_MU)

p_bins_full = np.linspace(0.0, pmax_mu*1.05, 24)
p_bins_low  = np.linspace(0.0, 0.8, 32)

def bq(x, y, bins): return binned_quantiles(x, y, bins, min_count=20)

stop_pc,  stop_q,  _ = bq(ev_hump.loc[stop_mask,'mu_p0'],  ev_hump.loc[stop_mask,'mu_cher_per_cm'],  p_bins_full)
punch_pc, punch_q, _ = bq(ev_hump.loc[punch_mask,'mu_p0'], ev_hump.loc[punch_mask,'mu_cher_per_cm'], p_bins_full)
stop_pc_z,  stop_q_z,  _ = bq(ev_hump.loc[stop_mask,'mu_p0'],  ev_hump.loc[stop_mask,'mu_cher_per_cm'],  p_bins_low)
punch_pc_z, punch_q_z, _ = bq(ev_hump.loc[punch_mask,'mu_p0'], ev_hump.loc[punch_mask,'mu_cher_per_cm'], p_bins_low)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('μ⁻ hump verification: stopping vs punch-through | Full 30k sample',
             fontsize=13, fontweight='bold')

for ax, pc_s, q_s, pc_p, q_p, p_g, title in [
    (axes[0], stop_pc, stop_q, punch_pc, punch_q, p_grid_mu,
     'A  Full momentum range'),
    (axes[1], stop_pc_z, stop_q_z, punch_pc_z, punch_q_z,
     p_grid_mu[p_grid_mu < 0.85], 'B  Zoom: 0–800 MeV/c'),
]:
    ax.fill_between(pc_s, q_s[0.16], q_s[0.84], alpha=0.2, color=C_MC)
    ax.plot(pc_s, q_s[0.50], 'o-', color=C_MC,  lw=1.8, ms=4, label='stopping  median')
    ax.fill_between(pc_p, q_p[0.16], q_p[0.84], alpha=0.2, color=C_MU2)
    ax.plot(pc_p, q_p[0.50], 's-', color=C_MU2, lw=1.8, ms=4, label='punch-through  median')
    ax.plot(p_g, frank_tamm_track_averaged(p_g, M_MU), '--', color='black', lw=2, label='Frank-Tamm')
    ax.axvline(PTHR_MU_EFF, color='grey', ls=':', lw=1.5, label=f'threshold p={PTHR_MU_EFF*1000:.0f} MeV/c')
    ax.set_xlabel('μ⁻ initial momentum (GeV/c)'); ax.set_ylabel('dN/dx (ph/cm)')
    ax.set_title(title); ax.legend(fontsize=9)

ax = axes[2]
ax.hist(ev_hump.loc[stop_mask,  'mu_cher_per_cm'].dropna(), bins=60,
        color=C_MC,  alpha=0.7, label=f'stopping  n={stop_mask.sum():,}')
ax.hist(ev_hump.loc[punch_mask, 'mu_cher_per_cm'].dropna(), bins=60,
        color=C_MU2, alpha=0.7, label=f'punch-through  n={punch_mask.sum():,}')
ax.axvline(FT_LIMIT, color='black', ls=':', lw=2, label=f'F-T β→1 = {FT_LIMIT:.0f} ph/cm')
ax.set_xlabel('dN/dx (ph/cm)'); ax.set_ylabel('Events')
ax.set_title('C  dN/dx distribution by containment'); ax.legend(fontsize=9)

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot7_muon_hump_stopping_vs_punchthrough.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 8 — Nuclear effects: π⁺+O inelastic hadronic inventory (3×3)

In [ ]:
# Load inelastic signal events from EventTree
with uproot.open(SHUF_PATH) as f:
    et_key  = sorted([k for k in f.keys() if 'EventTree' in k])[-1]
    ev_inel = f[et_key].arrays(
        ['event_id','mu_p0','pi_p0','mu_contained','pi_final_process',
         'mu_cher_total','pi_cher_total',
         'pi_x_end','pi_y_end','pi_z_end','pi_KE0'], library='pd')
ev_inel['mu_contained']    = ev_inel['mu_contained'].astype(bool)
ev_inel['pi_final_process']= (ev_inel['pi_final_process']
                              .astype(str).str.strip().str.rstrip('\x00'))
for c in ['mu_p0','pi_p0','mu_cher_total','pi_cher_total',
          'pi_x_end','pi_y_end','pi_z_end','pi_KE0']:
    ev_inel[c] = pd.to_numeric(ev_inel[c], errors='coerce')

sig_inel  = ev_inel[
    (ev_inel['mu_p0'] > MU_P_THRESHOLD) &
    (~ev_inel['mu_contained']) &
    ev_inel['pi_final_process'].str.contains('Inelastic', na=False)
].copy()
inel_ids  = set(sig_inel['event_id'].values)
print(f"Inelastic signal events: {len(sig_inel):,}")

# Load StepTree for inelastic events only
ST_COLS_IN = ['event_id','particle','parent_id','track_id',
              'from_primary_pi','cher_step','p_mag','KE',
              'x','y','z','time','process','step_num']
with uproot.open(SHUF_PATH) as f:
    st_key = sorted([k for k in f.keys() if 'StepTree' in k])[-1]
    st_inel_raw = f[st_key].arrays(ST_COLS_IN, library='pd')
st_inel_raw['particle']       = st_inel_raw['particle'].astype(str).str.strip().str.rstrip('\x00')
st_inel_raw['from_primary_pi']= st_inel_raw['from_primary_pi'].astype(bool)
for c in ['cher_step','p_mag','KE']:
    st_inel_raw[c] = pd.to_numeric(st_inel_raw[c], errors='coerce').fillna(0.)

st_ev     = st_inel_raw[st_inel_raw['event_id'].isin(inel_ids)].copy()
is_prim_pi= (st_ev['particle']=='pi+') & (st_ev['parent_id']==0)
is_had_sec = st_ev['from_primary_pi'] & ~is_prim_pi
first_steps= (st_ev[is_had_sec]
              .sort_values('step_num')
              .groupby(['event_id','track_id']).first().reset_index())

mass_map = {'pi+':0.13957,'pi-':0.13957,'proton':0.93827,'neutron':0.93957,
            'deuteron':1.87561,'triton':2.80892,'alpha':3.72742,
            'mu+':0.10566,'mu-':0.10566,'e+':0.000511,'e-':0.000511}
def p_thr(mass): return mass / np.sqrt(N_WATER**2 - 1)
first_steps['mass_GeV'] = first_steps['particle'].map(mass_map).fillna(1.0)
first_steps['p_thr']    = first_steps['mass_GeV'].apply(p_thr)
first_steps['above_thr']= first_steps['p_mag'] > first_steps['p_thr']

# Per-event Cherenkov from secondaries
def cher_by_species(pname):
    m = is_had_sec & (st_ev['particle']==pname)
    return st_ev[m].groupby('event_id')['cher_step'].sum().rename(f'cher_{pname}')
sec_cher = is_had_sec
sec_cher_sum = st_ev[is_had_sec].groupby('event_id')['cher_step'].sum().rename('sec_cher')
species_to_track = ['pi+','pi-','proton','e-','e+','mu+','deuteron','alpha','gamma']
cher_cols = [cher_by_species(s) for s in species_to_track]
sig_inel  = (sig_inel.set_index('event_id')
             .join(sec_cher_sum, how='left')
             .join(pd.concat(cher_cols, axis=1), how='left')
             .reset_index())
for col in ['sec_cher'] + [f'cher_{s}' for s in species_to_track]:
    if col in sig_inel.columns:
        sig_inel[col] = sig_inel[col].fillna(0.)

sig_inel['pi_r_end'] = np.sqrt(
    sig_inel['pi_x_end']**2 + sig_inel['pi_z_end']**2) / 10.  # mm→cm

fig = plt.figure(figsize=(18, 16))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle(f'π⁺+O Inelastic Interaction Products — Geant4 FTFP_BERT\n'
             f'Signal inelastic events n={len(sig_inel):,}',
             fontsize=13, fontweight='bold')

species_counts = first_steps['particle'].value_counts()
above_thr_cnt  = first_steps[first_steps['above_thr']]['particle'].value_counts()

ax = fig.add_subplot(gs[0, 0])
top_sp = species_counts.head(10)
ax.barh(top_sp.index[::-1], top_sp.values[::-1], color='steelblue')
ax.set_xlabel('Count (first step per track)'); ax.set_title('A  Secondary species produced')

ax = fig.add_subplot(gs[0, 1])
top_abv = above_thr_cnt.head(8)
ax.barh(top_abv.index[::-1], top_abv.values[::-1], color='darkorange')
ax.set_xlabel('Count'); ax.set_title('B  Species above Cherenkov threshold')

ax = fig.add_subplot(gs[0, 2])
ax.hist(sig_inel['pi_KE0'], bins=50, color=C_PI, alpha=0.75)
ax.set_xlabel('π⁺ KE at inelastic vertex (GeV)'); ax.set_ylabel('Events')
ax.set_title('C  π⁺ kinetic energy at interaction point')

ax = fig.add_subplot(gs[1, 0])
ax.hist(sig_inel['sec_cher'].dropna(), bins=50, color=C_INEL, alpha=0.75)
ax.axvline(sig_inel['sec_cher'].mean(), ls='--', lw=1.8, color='black',
           label=f'mean={sig_inel["sec_cher"].mean():.0f}')
ax.set_xlabel('Total Cherenkov from secondaries'); ax.set_ylabel('Events')
ax.set_title('D  Secondary Cherenkov yield per event'); ax.legend(fontsize=9)

ax = fig.add_subplot(gs[1, 1])
ax.hist(sig_inel['pi_cher_total'], bins=50, color=C_PI, alpha=0.6, label='π⁺ primary')
ax.hist(sig_inel['sec_cher'].dropna(), bins=50, color=C_INEL, alpha=0.6, label='secondaries')
ax.set_xlabel('Cherenkov photons'); ax.set_ylabel('Events')
ax.set_title('E  Primary π⁺ vs secondary Cherenkov'); ax.legend(fontsize=9)

ax = fig.add_subplot(gs[1, 2])
ax.hist(sig_inel['pi_r_end'].dropna(), bins=50, color=C_OTHER, alpha=0.75)
ax.axvline(160., color='black', ls='--', lw=1.8, label='R = 160 cm wall')
ax.set_xlabel('Inelastic vertex radius (cm)'); ax.set_ylabel('Events')
ax.set_title('F  Inelastic vertex radial position'); ax.legend(fontsize=9)

cher_species_means = {s: sig_inel[f'cher_{s}'].mean() for s in species_to_track if f'cher_{s}' in sig_inel}
top_cher = sorted(cher_species_means.items(), key=lambda x: x[1], reverse=True)[:6]
ax = fig.add_subplot(gs[2, 0])
labels_c = [t[0] for t in top_cher]
vals_c   = [t[1] for t in top_cher]
ax.bar(labels_c, vals_c, color='teal', alpha=0.8)
ax.set_ylabel('Mean Cherenkov / event'); ax.set_title('G  Cherenkov per species (top 6)')

ax = fig.add_subplot(gs[2, 1])
ax.hist2d(sig_inel['pi_KE0'], sig_inel['sec_cher'].fillna(0),
          bins=40, cmap='viridis')
ax.set_xlabel('π⁺ KE at vertex (GeV)'); ax.set_ylabel('Secondary Cherenkov')
ax.set_title('H  Secondary Cher vs π⁺ KE at vertex')

ax = fig.add_subplot(gs[2, 2])
ax.scatter(sig_inel['pi_cher_total'], sig_inel['sec_cher'].fillna(0),
           s=4, alpha=0.3, color=C_PI)
ax.set_xlabel('π⁺ primary Cherenkov'); ax.set_ylabel('Secondary Cherenkov')
ax.set_title('I  Primary vs secondary Cherenkov (scatter)')

# fig.savefig(PNG_DIR / 'plot8_nuclear_inelastic_inventory.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 9 — Radioactive isotope inventory from π⁺+O spallation

In [ ]:
BETA_PLUS_ISOTOPES = {
    'N12': (0.011,  16.32, 'β⁺'),
    'B8':  (0.770,  13.70, 'β⁺'),
    'C10': (19.3,    1.47, 'β⁺'),
    'O15': (122.,    1.72, 'β⁺'),
    'N13': (597.,    1.19, 'β⁺'),
    'C11': (1224.,   0.96, 'β⁺'),
    'F17': (64.5,    1.74, 'β⁺'),
    'O14': (70.6,    1.81, 'β⁺'),
}

# Production inventory from StepTree (search for these nuclei in inelastic events)
with uproot.open(SHUF_PATH) as f:
    st_key  = sorted([k for k in f.keys() if 'StepTree' in k])[-1]
    st_iso  = f[st_key].arrays(
        ['event_id','particle','parent_id','from_primary_pi','p_mag','KE','time'],
        library='pd')
st_iso['particle']       = st_iso['particle'].astype(str).str.strip().str.rstrip('\x00')
st_iso['from_primary_pi']= st_iso['from_primary_pi'].astype(bool)

st_iso_ev = st_iso[st_iso['event_id'].isin(inel_ids)].copy()
iso_names = list(BETA_PLUS_ISOTOPES.keys())
iso_counts = {}
for iso in iso_names:
    mask = (st_iso_ev['particle'] == iso) & (st_iso_ev['from_primary_pi'])
    iso_counts[iso] = st_iso_ev[mask]['event_id'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Radioactive β⁺ Isotope Production — π⁺+O Inelastic | FTFP_BERT',
             fontsize=13, fontweight='bold')

# Panel A: production counts
ax = axes[0]
sorted_iso = sorted(iso_counts.items(), key=lambda x: x[1], reverse=True)
labels_iso = [k for k, v in sorted_iso]
vals_iso   = [v for k, v in sorted_iso]
colors_iso = ['crimson' if BETA_PLUS_ISOTOPES[iso][1] > 5 else
              'darkorange' if BETA_PLUS_ISOTOPES[iso][1] > 1 else 'steelblue'
              for iso in labels_iso]
ax.bar(labels_iso, vals_iso, color=colors_iso, alpha=0.8)
ax.set_xlabel('Isotope'); ax.set_ylabel('Events with at least one production')
ax.set_title('A  β⁺ emitter production counts (events)\n'
             'Red = E_β>5 MeV energetic, orange = Michel-like 0.5–5 MeV')
ax.tick_params(axis='x', rotation=30)

# Panel B: β⁺ endpoint energy vs half-life scatter
ax = axes[1]
for iso, (t_half, E_end, _) in BETA_PLUS_ISOTOPES.items():
    color = ('crimson'     if E_end > 5 else
             'darkorange'  if 0.5 < E_end < 5 else
             'steelblue')
    ax.scatter(t_half, E_end, s=iso_counts.get(iso, 0)*0.5+30,
               color=color, alpha=0.8, zorder=3)
    ax.annotate(iso, (t_half, E_end), fontsize=9,
                xytext=(5, 3), textcoords='offset points')
ax.axhline(52.8, color='grey', ls=':', lw=1.5, label='μ⁺ decay endpoint ~52.8 MeV')
ax.axhspan(0.5, 60., alpha=0.05, color='green', label='Michel-tag window rough range')
ax.set_xscale('log')
ax.set_xlabel('Half-life (s)'); ax.set_ylabel('β⁺ endpoint energy (MeV)')
ax.set_title('B  Isotope endpoint vs half-life\n(marker size ∝ production rate)')
ax.legend(fontsize=9)

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot9_isotope_inventory.png', dpi=600, bbox_inches='tight')
plt.show()

print('\nβ⁺ isotope production summary:')
print(f'{"Isotope":8s} {"t½":>10s} {"E_β (MeV)":>12s} {"Events":>8s} {"Concern"}')
print('─'*55)
for iso, (t_half, E_end, _) in sorted(BETA_PLUS_ISOTOPES.items(), key=lambda x: x[1][0]):
    t_str   = f'{t_half*1000:.0f} ms' if t_half<1 else (f'{t_half:.1f} s' if t_half<60 else
              f'{t_half/60:.1f} min' if t_half<3600 else f'{t_half/3600:.1f} hr')
    concern = ('⚠ ENERGETIC'   if E_end > 5 else
               '⚠ MICHEL-LIKE' if 0.5 < E_end < 5 else 'low energy')
    print(f'  {iso:6s} {t_str:>10s}  {E_end:>8.2f} MeV  {iso_counts.get(iso,0):>6d}  {concern}')

---
## Plot 10 — Decay timing: full waveform prompt + delayed (log-log)

In [ ]:
# Accumulate log-spaced PMT hit-time histograms from StepTree
LOG_BINS   = np.logspace(np.log10(1), np.log10(20000), 200)  # 1 ns → 20 µs
LOG_CENTS  = np.sqrt(LOG_BINS[:-1] * LOG_BINS[1:])
LOG_WIDTHS = np.diff(LOG_BINS)

acc_log = {k: np.zeros(len(LOG_CENTS)) for k in ['mu','pi','smu','michel']}
n_log   = {k: 0 for k in acc_log}

st_cols_decay = ['event_id','particle','parent_id','track_id',
                 'x','y','z','time','cher_step','from_primary_pi']

# Need mu+ track map first
print('Pass 1: Building mu+ track map...')
mu_plus_map = {}
with uproot.open(SHUF_PATH) as f:
    st_key = sorted([k for k in f.keys() if 'StepTree' in k])[-1]
    for chunk in f[st_key].iterate(
            ['event_id','particle','track_id','from_primary_pi'],
            step_size=500_000, library='pd'):
        chunk['particle']        = chunk['particle'].astype(str).str.strip().str.rstrip('\x00')
        chunk['from_primary_pi'] = chunk['from_primary_pi'].astype(bool)
        mup = chunk[(chunk['particle']=='mu+') & chunk['from_primary_pi']
                    & chunk['event_id'].isin(decay_ids)]
        for eid, grp in mup.groupby('event_id'):
            if eid not in mu_plus_map: mu_plus_map[eid] = set()
            mu_plus_map[eid].update(grp['track_id'].tolist())
print(f'  mu+ tracks in {len(mu_plus_map):,} decay events')

print('Pass 2: Accumulating log-spaced hit times...')
with uproot.open(SHUF_PATH) as f:
    st_key = sorted([k for k in f.keys() if 'StepTree' in k])[-1]
    for chunk in f[st_key].iterate(st_cols_decay, step_size=500_000, library='pd'):
        chunk['particle']        = chunk['particle'].astype(str).str.strip().str.rstrip('\x00')
        chunk['from_primary_pi'] = chunk['from_primary_pi'].astype(bool)
        chunk = chunk[(chunk['event_id'].isin(decay_ids)) & (chunk['cher_step'] > 0)]
        if len(chunk) == 0: continue
        d_wall      = tof_to_cylinder(chunk['x'].values, chunk['y'].values, chunk['z'].values)
        chunk = chunk.copy()
        chunk['t_hit']    = chunk['time'] + d_wall / C_WATER
        chunk['cher_det'] = chunk['cher_step'] * DET_EFF
        is_mu  = (chunk['particle']=='mu-') & (chunk['parent_id']==0)
        is_pi  = (chunk['particle']=='pi+') & chunk['from_primary_pi']
        is_smu = (chunk['particle']=='mu+') & chunk['from_primary_pi']
        ep     = chunk[(chunk['particle']=='e+') & chunk['from_primary_pi']].copy()
        if len(ep):
            m_idx    = np.array([pid in mu_plus_map.get(eid, set())
                                 for pid, eid in zip(ep['parent_id'].values, ep['event_id'].values)])
            is_michel_idx = ep.index[m_idx]
        else:
            is_michel_idx = pd.Index([])
        for key, sub in [('mu', chunk[is_mu]), ('pi', chunk[is_pi]),
                         ('smu', chunk[is_smu]),
                         ('michel', chunk.loc[is_michel_idx] if len(is_michel_idx) else chunk.iloc[0:0])]:
            if len(sub) == 0: continue
            t  = np.clip(sub['t_hit'].values, LOG_BINS[0], None)
            w  = sub['cher_det'].values
            acc_log[key] += np.histogram(t, bins=LOG_BINS, weights=w)[0]
            n_log[key]   += 1

n_ev = max(n_log['mu'], 1)
rate_mu     = acc_log['mu']    / (n_ev * LOG_WIDTHS)
rate_pi     = acc_log['pi']    / (n_ev * LOG_WIDTHS)
rate_smu    = acc_log['smu']   / (n_ev * LOG_WIDTHS)
rate_michel = acc_log['michel']/ (n_ev * LOG_WIDTHS)

fig, ax = plt.subplots(figsize=(12, 6))
ax.loglog(LOG_CENTS, np.where(rate_mu>0,    rate_mu,    1e-12), color=C_MU,    lw=2, label='μ⁻ primary')
ax.loglog(LOG_CENTS, np.where(rate_pi>0,    rate_pi,    1e-12), color=C_PI,    lw=2, label='π⁺ primary')
ax.loglog(LOG_CENTS, np.where(rate_smu>0,   rate_smu,   1e-12), color=C_MU2,  lw=2, label='μ⁺ (decay product)')
ax.loglog(LOG_CENTS, np.where(rate_michel>0,rate_michel,1e-12), color=C_DECAY, lw=2, label='Michel e⁺')
ax.axvline(2197., color='grey', ls=':', lw=1.5, label='μ⁺ lifetime = 2197 ns')
ax.set_xlabel('PMT wall-hit time (ns)')
ax.set_ylabel('Avg detected ph / event / ns')
ax.set_title('Decay timing: full ANNIE waveform — prompt + delayed | Decay events only',
             fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(1, 20000)
plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot10_decay_timing_loglog.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 11 — Prompt shoulder: pion vs muon timing (0–60 ns)

In [ ]:
T_PROMPT   = np.linspace(0, 60, 241)
T_PROMPT_C = (T_PROMPT[:-1] + T_PROMPT[1:]) / 2
sum_mu_p   = np.zeros(len(T_PROMPT_C))
sum_pi_p   = np.zeros(len(T_PROMPT_C))
n_mu_ev    = 0; n_pi_ev = 0
pion_outlives_ids = set(np.int32(list(
    ev_sig[ev_sig['pi_time_end'] > ev_sig['mu_water_exit_ns']].index)))
sum_mu_out = np.zeros(len(T_PROMPT_C))
sum_pi_out = np.zeros(len(T_PROMPT_C))
n_out = 0

st_cols_sh = ['event_id','particle','parent_id','x','y','z','time','cher_step','from_primary_pi']
print(f"Pion-outlives-muon events: {len(pion_outlives_ids):,} "
      f"({len(pion_outlives_ids)/len(ev_sig)*100:.1f}%)")

with uproot.open(SHUF_PATH) as f:
    st_key = sorted([k for k in f.keys() if 'StepTree' in k])[-1]
    for i, chunk in enumerate(f[st_key].iterate(st_cols_sh, step_size=500_000, library='pd')):
        chunk['particle']        = chunk['particle'].astype(str).str.strip().str.rstrip('\x00')
        chunk['from_primary_pi'] = chunk['from_primary_pi'].astype(bool)
        mask = (
            chunk['event_id'].isin(sig_ids) & (chunk['cher_step'] > 0) &
            (((chunk['particle']=='mu-') & (chunk['parent_id']==0)) |
             (chunk['particle']=='pi+'))
        )
        chunk = chunk[mask].copy()
        if len(chunk) == 0: continue
        chunk = chunk.merge(mu_exit_lookup, left_on='event_id', right_index=True, how='left')
        d_wall = tof_to_cylinder(chunk['x'].values, chunk['y'].values, chunk['z'].values)
        chunk['t_hit']   = chunk['time'] + d_wall / C_WATER
        chunk['cher_det']= chunk['cher_step'] * DET_EFF
        is_mu = (chunk['particle']=='mu-') & (chunk['parent_id']==0)
        is_pi = (chunk['particle']=='pi+')
        for eid, grp in chunk[is_mu].groupby('event_id'):
            sum_mu_p += np.histogram(grp['t_hit'], bins=T_PROMPT, weights=grp['cher_det'])[0]; n_mu_ev+=1
        for eid, grp in chunk[is_pi].groupby('event_id'):
            sum_pi_p += np.histogram(grp['t_hit'], bins=T_PROMPT, weights=grp['cher_det'])[0]; n_pi_ev+=1
        out_mu = chunk[is_mu & chunk['event_id'].isin(pion_outlives_ids)]
        out_pi = chunk[is_pi & chunk['event_id'].isin(pion_outlives_ids)]
        for eid, grp in out_mu.groupby('event_id'):
            sum_mu_out += np.histogram(grp['t_hit'], bins=T_PROMPT, weights=grp['cher_det'])[0]
        for eid, grp in out_pi.groupby('event_id'):
            sum_pi_out += np.histogram(grp['t_hit'], bins=T_PROMPT, weights=grp['cher_det'])[0]; n_out+=1
        if (i+1)%10==0: print(f'  chunk {i+1}')

avg_mu_p    = sum_mu_p    / max(n_mu_ev, 1)
avg_pi_p    = sum_pi_p    / max(n_pi_ev, 1)
avg_mu_out  = sum_mu_out  / max(n_out, 1)
avg_pi_out  = sum_pi_out  / max(n_out, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Prompt shoulder: μ⁻ vs π⁺ timing | 0–60 ns | Signal sample',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(T_PROMPT_C, avg_mu_p, color=C_MU, lw=2, label=f'μ⁻ (n={n_mu_ev:,})')
ax.plot(T_PROMPT_C, avg_pi_p, color=C_PI, lw=2, label=f'π⁺ (n={n_pi_ev:,})')
ax.set_xlabel('PMT hit time (ns)'); ax.set_ylabel('Avg detected ph/event/bin')
ax.set_title('A  All signal events — linear scale'); ax.legend(fontsize=9)

ax = axes[1]
ax.semilogy(T_PROMPT_C, np.where(avg_mu_p>0, avg_mu_p, 1e-9), color=C_MU, lw=2, label='μ⁻')
ax.semilogy(T_PROMPT_C, np.where(avg_pi_p>0, avg_pi_p, 1e-9), color=C_PI, lw=2, label='π⁺')
ax.semilogy(T_PROMPT_C, np.where(avg_mu_out>0, avg_mu_out, 1e-9), color=C_MU, lw=1.5, ls='--',
            label=f'μ⁻ pion-outlives subset (n={n_out:,})')
ax.semilogy(T_PROMPT_C, np.where(avg_pi_out>0, avg_pi_out, 1e-9), color=C_PI, lw=1.5, ls='--',
            label='π⁺ pion-outlives subset')
ax.set_xlabel('PMT hit time (ns)'); ax.set_ylabel('Avg detected ph/event/bin')
ax.set_title('B  Log scale + pion-outlives-muon subset'); ax.legend(fontsize=9)

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot11_prompt_shoulder.png', dpi=600, bbox_inches='tight')
plt.show()

---
## Plot 12 — High-resolution prompt zoom (4–12 ns, event 23630 removed)

In [ ]:
OUTLIER_EIDS  = {LATE_OUTLIER_EID}
ev_sig_plot   = ev_sig_no23630.copy()
if 'event_id' not in ev_sig_plot.columns:
    ev_sig_plot['event_id'] = ev_sig_plot.index
ev_sig_plot['event_id'] = pd.to_numeric(ev_sig_plot['event_id'], errors='coerce')
ev_sig_plot = ev_sig_plot.dropna(subset=['event_id']).copy()
ev_sig_plot['event_id'] = ev_sig_plot['event_id'].astype(np.int32)
ev_sig_plot = ev_sig_plot.set_index('event_id', drop=False)
print(f'Plotting sample: {len(ev_sig_plot):,} events')

T_ZOOM    = np.arange(4.0, 12.0 + 0.1, 0.1)
T_ZOOM_C  = 0.5 * (T_ZOOM[:-1] + T_ZOOM[1:])
sum_mu_z  = np.zeros(len(T_ZOOM_C))
sum_pi_ch = np.zeros(len(T_ZOOM_C))  # chain-based
sum_pi_st = np.zeros(len(T_ZOOM_C))  # strict primary
sig_ids_z = set(np.int32(ev_sig_plot.index.tolist()))
pi_tid_lk = ev_sig_plot['pi_track_id'].to_dict()
n_mu_z    = 0; n_pi_ch_z = 0; n_pi_st_z = 0

zoom_cols = ['event_id','particle','parent_id','track_id','x','y','z','time','cher_step','from_primary_pi']
with uproot.open(SHUF_PATH) as f:
    st_key = sorted([k for k in f.keys() if 'StepTree' in k])[-1]
    for i, chunk in enumerate(f[st_key].iterate(zoom_cols, step_size=500_000, library='pd')):
        chunk['particle']        = chunk['particle'].astype(str).str.strip().str.rstrip('\x00')
        chunk['from_primary_pi'] = chunk['from_primary_pi'].astype(bool)
        chunk = chunk[(chunk['event_id'].isin(sig_ids_z)) & (chunk['cher_step']>0)].copy()
        if len(chunk)==0: continue
        d_wall = tof_to_cylinder(chunk['x'].values, chunk['y'].values, chunk['z'].values)
        chunk['t_hit']   = chunk['time'] + d_wall / C_WATER
        chunk['cher_det']= chunk['cher_step'] * DET_EFF
        mu      = chunk[(chunk['particle']=='mu-') & (chunk['parent_id']==0)]
        pi_ch   = chunk[(chunk['particle']=='pi+') & (chunk['from_primary_pi'])].copy()
        pi_st   = chunk[chunk['particle']=='pi+'].copy()
        if len(pi_st):
            pi_st['primary_tid'] = pi_st['event_id'].map(pi_tid_lk)
            pi_st = pi_st[pi_st['track_id']==pi_st['primary_tid']]
        for _, grp in mu.groupby('event_id'):
            sum_mu_z  += np.histogram(grp['t_hit'], bins=T_ZOOM, weights=grp['cher_det'])[0]; n_mu_z+=1
        for _, grp in pi_ch.groupby('event_id'):
            sum_pi_ch += np.histogram(grp['t_hit'], bins=T_ZOOM, weights=grp['cher_det'])[0]; n_pi_ch_z+=1
        for _, grp in pi_st.groupby('event_id'):
            sum_pi_st += np.histogram(grp['t_hit'], bins=T_ZOOM, weights=grp['cher_det'])[0]; n_pi_st_z+=1
        if (i+1)%10==0: print(f'  chunk {i+1}')

avg_mu_z  = sum_mu_z  / max(n_mu_z,    1)
avg_pi_ch = sum_pi_ch / max(n_pi_ch_z, 1)
avg_pi_st = sum_pi_st / max(n_pi_st_z, 1)

def unit_area(y):
    s = np.nansum(y); return y/s if s>0 else y

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('High-res prompt zoom (4–12 ns) | Event 23630 removed | Signal sample',
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(T_ZOOM_C, avg_mu_z,  label='μ⁻ primary',        lw=2, color=C_MU)
ax.plot(T_ZOOM_C, avg_pi_ch, label='π⁺ chain-based',    lw=2, color=C_PI)
ax.plot(T_ZOOM_C, avg_pi_st, label='π⁺ strict primary', lw=2, color=C_DECAY, ls='--')
ax.set_xlabel('Wall hit time (ns)'); ax.set_ylabel('Avg detected ph/event/0.1 ns bin')
ax.set_title('A  Absolute yield — linear scale'); ax.legend(fontsize=9)

ax = axes[1]
ax.plot(T_ZOOM_C, unit_area(avg_mu_z),  lw=2, color=C_MU,   label='μ⁻ primary')
ax.plot(T_ZOOM_C, unit_area(avg_pi_ch), lw=2, color=C_PI,   label='π⁺ chain-based')
ax.plot(T_ZOOM_C, unit_area(avg_pi_st), lw=2, color=C_DECAY, ls='--', label='π⁺ strict primary')
ax.set_xlabel('Wall hit time (ns)'); ax.set_ylabel('Unit-area normalized shape')
ax.set_title('B  Timing shape only (unit-area normalized)'); ax.legend(fontsize=9)

plt.tight_layout()
# fig.savefig(PNG_DIR / 'plot12_prompt_zoom_hires.png', dpi=600, bbox_inches='tight')
plt.show()

print('='*60)
print('HIGH-RES PROMPT ZOOM SUMMARY')
print('='*60)
print(f'μ⁻ peak time               : {T_ZOOM_C[np.nanargmax(avg_mu_z)]:.2f} ns')
print(f'π⁺ chain-based peak time   : {T_ZOOM_C[np.nanargmax(avg_pi_ch)]:.2f} ns')
print(f'π⁺ strict primary peak time: {T_ZOOM_C[np.nanargmax(avg_pi_st)]:.2f} ns')
print(f'μ⁻ integrated (4–12 ns)    : {np.nansum(avg_mu_z):.3f}')
print(f'π⁺ chain-based (4–12 ns)   : {np.nansum(avg_pi_ch):.3f}')
print(f'π⁺ strict primary (4–12 ns): {np.nansum(avg_pi_st):.3f}')